### Inference: Parsimony (CI / RI / RCI)

`toytree.infer.consistency_and_retention_indices` summarizes how well a
single discrete trait follows a tree topology.

The result is returned as a pandas DataFrame with one row for each
statistic (`fitch_parsimony_score`, `CI`, `RI`, and `RCI`). Columns
report the observed value, the permutation null mean, one-sided
p-values in both directions, and the tail that is usually interpreted as
phylogenetic signal for that statistic.

The **consistency index (CI)** compares the minimum possible number of
changes for the observed number of states to the observed Fitch
parsimony score. Lower CI means more extra changes, so it is commonly
used as a measure of homoplasy.

The **retention index (RI)** measures how well trait states are retained
as clade-structured patterns instead of being scattered repeatedly across
the tree. High RI means the observed states can be explained largely by
shared ancestry rather than repeated gains or losses.

The **rescaled consistency index (RCI)** is `CI * RI`. It combines the
homoplasy penalty from CI with the retention term from RI, which often
makes it more useful for comparing characters across trees or datasets
than CI alone.


In [ ]:
import numpy as np
import pandas as pd
import toytree

### How to read the result table

The returned DataFrame includes both one-sided permutation p-values for
each statistic:

- `p_value_greater`: tests whether the observed value is greater than the
  permutation null.
- `p_value_less`: tests whether the observed value is smaller than the
  permutation null.

The `signal_tail` column indicates which direction is usually interpreted
as stronger phylogenetic structure. For `CI`, `RI`, and `RCI`, larger
values indicate stronger tree-structured signal. For
`fitch_parsimony_score`, the direction is reversed: fewer implied
changes indicate stronger phylogenetic structure.


### Demonstration

First simulate a small tree and one discrete trait with three states.


In [ ]:
tree = toytree.rtree.unittree(20, seed=42)
trait = tree.pcm.simulate_discrete_trait(
    nstates=3,
    tips_only=True,
    state_names="ABC",
    seed=7,
)
trait.name = "state"
trait.head(10)

In [ ]:
trait.value_counts().sort_index()

Now compute the parsimony summary table for the observed trait.


In [ ]:
signal_stats = toytree.infer.consistency_and_retention_indices(
    tree,
    trait,
    npermutations=500,
    rng=7,
)
signal_stats

You can inspect whichever tail matters for your question directly
from the result table. The snippet below focuses on the three homoplasy
indices and shows both one-sided p-values alongside the recommended
`signal_tail` direction.


In [ ]:
signal_stats.loc[
    ["CI", "RI", "RCI"],
    ["observed", "null_mean", "p_value_greater", "p_value_less", "signal_tail"],
]


To make the interpretation concrete, compare that clustered trait to a
randomized version with the same state counts but shuffled tip labels.


In [ ]:
rng = np.random.default_rng(7)
randomized_trait = pd.Series(
    rng.permutation(trait.to_numpy()),
    index=trait.index,
    name="state",
    dtype=object,
)
randomized_trait.head(10)

In [ ]:
randomized_stats = toytree.infer.consistency_and_retention_indices(
    tree,
    randomized_trait,
    npermutations=500,
    rng=7,
)
comparison = pd.DataFrame(
    {
        "clustered_trait": signal_stats["observed"],
        "randomized_trait": randomized_stats["observed"],
    }
)
comparison


In this example the clustered trait has a lower Fitch parsimony score
and higher CI, RI, and RCI than the randomized trait. That is the
typical pattern when a character is more consistent with the tree than a
random tip-label permutation.
